# Importing and Loading Data

In [1]:
import pandas as pd

# The file has two sheets (2009-10 and 2010-11). sheet_name=None loads both.
sheets = pd.read_excel(r"C:/Users/49dhe/Desktop/project 2/1_DATA/online_retail_II.xlsx", sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


# Checking Data quality

In [2]:
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Duplicate rows: 34335


# Cleaning Data

In [3]:
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

df = df.drop_duplicates()
print("After removing duplicates:", len(df))

df = df[~df["Invoice"].str.startswith("C")]      # C = cancelled invoice
print("After removing cancellations:", len(df))

df = df.dropna(subset=["Customer ID"])            # RFM needs a customer
print("After removing missing customers:", len(df))

df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print("After removing zero prices:", len(df))

non_products = ["POST", "M", "C2", "ADJUST", "BANK CHARGES", "PADS",
                "DOT", "TEST001", "D", "ADJUST2", "SP1002", "TEST002"]
df = df[~df["StockCode"].isin(non_products)]      # postage, fees, tests
print("After removing non-product codes:", len(df))

After removing duplicates: 1033036
After removing cancellations: 1013932
After removing missing customers: 779495
After removing zero prices: 779425
After removing non-product codes: 776577


# Revenue and summaries

In [4]:
df["Customer ID"] = df["Customer ID"].astype(int)
df["Revenue"] = df["Quantity"] * df["Price"]

print("Customers:", df["Customer ID"].nunique())
print("Invoices:", df["Invoice"].nunique())
print("Revenue: £", round(df["Revenue"].sum(), 2))
print("Dates:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
print(df["Country"].value_counts(normalize=True).head(3))

Customers: 5852
Invoices: 36594
Revenue: £ 17068567.97
Dates: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Country
United Kingdom    0.900887
Germany           0.020329
EIRE              0.019765
Name: proportion, dtype: float64


# Save cleaned data

In [5]:
df.to_csv(r"C:/Users/49dhe/Desktop/project 2/1_DATA/retail_clean.csv", index=False)